# Análise das Exportações Brasileiras para a França (2016–2020)

Análise exploratória dos dados de exportação do Brasil para a França: evolução anual, categorias de produto mais exportadas, cidades que mais exportaram em 2020 e o peso do cacau e do chocolate na pauta.

**Dados:** `chocolate_sales.csv` (valores em US$ FOB).

## 1. Carregando e inspecionando os dados

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

tabela = pd.read_csv('chocolate_sales.csv')
tabela.info()

## 2. Tratando a duplicidade de registros

Cada registro de exportação aparece **duas vezes** na base: uma com o bloco econômico `Europe` e outra com `European Union (EU)`. Somar tudo sem tratar isso contaria todos os valores em dobro, então mantemos apenas uma cópia de cada registro.

In [ ]:
print(tabela['Economic Block'].value_counts())

# Todas as colunas, exceto o bloco econômico, identificam o registro
colunas_chave = [c for c in tabela.columns if c != 'Economic Block']
repeticoes = tabela.groupby(colunas_chave).size()
print('\nNúmero de vezes que cada registro aparece:', repeticoes.unique().tolist())

# Mantém uma única cópia de cada registro (bloco da União Europeia)
tabela = tabela[tabela['Economic Block'] == 'European Union (EU)'].copy()
print(f'\nLinhas após remover a duplicidade: {len(tabela):,}')

In [ ]:
# Funções auxiliares.
# Os números continuam numéricos na base; a formatação em US$ é aplicada só na exibição.
def formatar(valor):
    return f'US${valor:,.0f}'

def mostrar(tabela_numerica):
    return tabela_numerica.style.format({'US$ FOB': formatar})

## 3. Evolução das exportações ao longo dos anos

In [ ]:
# Valor exportado por ano e variação percentual em relação ao ano anterior
exportacao_anual = tabela.groupby('Year')[['US$ FOB']].sum()
exportacao_anual['Variação (%)'] = exportacao_anual['US$ FOB'].pct_change() * 100

exportacao_anual.style.format({'US$ FOB': formatar, 'Variação (%)': '{:+.1f}%'}, na_rep='-')

In [ ]:
ax = (exportacao_anual['US$ FOB'] / 1e9).plot(kind='line', marker='o', figsize=(8, 4))
ax.set_title('Exportações por ano (US$ bilhões)')
ax.set_xlabel('Ano')
ax.set_ylabel('US$ bilhões')
ax.set_xticks(exportacao_anual.index)
ax.grid(alpha=0.3)
plt.show()

## 4. Categorias de produto que mais venderam

In [ ]:
# Total exportado por categoria de produto (SH2), do maior para o menor
produtos_exportados = (
    tabela.groupby('SH2 Description')[['US$ FOB']].sum()
    .sort_values('US$ FOB', ascending=False)
)

# Mostrando as 10 maiores categorias
mostrar(produtos_exportados.head(10))

In [ ]:
top_produtos = produtos_exportados.head(10).iloc[::-1]  # inverte para o maior ficar no topo do gráfico

ax = (top_produtos['US$ FOB'] / 1e9).plot(kind='barh', figsize=(10, 5))
ax.set_yticklabels([d[:45] + '...' if len(d) > 45 else d for d in top_produtos.index])
ax.set_title('10 categorias mais exportadas, 2016–2020 (US$ bilhões)')
ax.set_xlabel('US$ bilhões')
ax.set_ylabel('')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Cidades que mais exportaram em 2020

In [ ]:
tabela_2020 = tabela[tabela['Year'] == 2020]

cidades_2020 = (
    tabela_2020.groupby('City')[['US$ FOB']].sum()
    .sort_values('US$ FOB', ascending=False)
)

# Mostrando as 10 maiores cidades
mostrar(cidades_2020.head(10))

In [ ]:
top_cidades = cidades_2020.head(10).iloc[::-1]

ax = (top_cidades['US$ FOB'] / 1e6).plot(kind='barh', figsize=(9, 5))
ax.set_title('10 cidades que mais exportaram em 2020 (US$ milhões)')
ax.set_xlabel('US$ milhões')
ax.set_ylabel('')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Produtos mais vendidos pelas 2 cidades que mais exportaram (2020)

In [ ]:
# Repete a análise de produtos para cada uma das 2 maiores cidades exportadoras de 2020
for cidade in cidades_2020.index[:2]:
    produtos_cidade = (
        tabela_2020[tabela_2020['City'] == cidade]
        .groupby('SH2 Description')[['US$ FOB']].sum()
        .sort_values('US$ FOB', ascending=False)
    )
    print(f'\n{cidade}: 10 produtos mais exportados em 2020')
    display(mostrar(produtos_cidade.head(10)))

## 7. Cacau e chocolate (SH2 18)

Como o projeto parte de uma empresa de chocolate, isolamos a categoria de cacau e suas preparações (SH2 18), que inclui o chocolate propriamente dito (SH4 1806).

In [ ]:
cacau = tabela[tabela['SH2 Code'] == 18]

total_cacau = cacau['US$ FOB'].sum()
total_geral = tabela['US$ FOB'].sum()
print(f'Cacau e preparações: {formatar(total_cacau)} ({total_cacau / total_geral:.2%} do total exportado para a França)')

# Detalhe por produto (SH4)
por_produto = (
    cacau.groupby(['SH4 Code', 'SH4 Description'])[['US$ FOB']].sum()
    .sort_values('US$ FOB', ascending=False)
)
mostrar(por_produto)

In [ ]:
cacau_anual = cacau.groupby('Year')[['US$ FOB']].sum()

ax = (cacau_anual['US$ FOB'] / 1e6).plot(kind='bar', figsize=(8, 4), rot=0)
ax.set_title('Exportações de cacau e preparações por ano (US$ milhões)')
ax.set_xlabel('Ano')
ax.set_ylabel('US$ milhões')
ax.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# Cidades que mais exportaram cacau e preparações (2016–2020)
cidades_cacau = (
    cacau.groupby('City')[['US$ FOB']].sum()
    .sort_values('US$ FOB', ascending=False)
)
mostrar(cidades_cacau.head(5))